# Racial Bias in Facial Recognition
### An Analysis Using EfficientNet Transfer Learning + Grad-CAM

This notebook trains a facial recognition model on the [FairFace dataset](https://github.com/joojs/fairface) — ~108,000 faces balanced across 7 racial groups — and analyzes where and how the model's accuracy degrades across those groups.

**The goal is not to build a race classifier.** The goal is to expose *bias disparity*: the unequal accuracy a model exhibits across demographic groups, mirroring the failures documented in commercial facial recognition systems used by law enforcement.

---

### What changed from the original project

| Original (Dec 2021) | This version |
|---|---|
| Custom 2-layer CNN trained from scratch | EfficientNetB0 pretrained on ImageNet |
| ~7,000 training images | Full FairFace dataset (~86,000 train images) |
| 140-image test set | Full validation set (~13,000 images) |
| Accuracy + confusion matrix only | Per-class F1 / precision / recall + Grad-CAM |
| Ran on local laptop (slow) | Google Colab (GPU) |

The original model's core finding — that even a balanced dataset produces biased predictions — is tested again here at larger scale with a stronger architecture.

In [ ]:
# ── Google Drive (persists images between Colab sessions) ─────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Kaggle API setup (used to download FairFace) ──────────────────────────
# 1. Go to kaggle.com → Settings → API → Create New Token
# 2. Upload the downloaded kaggle.json when prompted below
!pip install -q kaggle
from google.colab import files
files.upload()  # upload kaggle.json

import os
os.makedirs('/root/.config/kaggle', exist_ok=True)
!mv kaggle.json /root/.config/kaggle/
!chmod 600 /root/.config/kaggle/kaggle.json

# ── Download FairFace to Drive (skips if already downloaded) ──────────────
DATA_DIR = '/content/drive/MyDrive/fairface'
os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(f'{DATA_DIR}/train'):
    print("Downloading FairFace (~3.5 GB)...")
    !kaggle datasets download -d jessicali9530/fairface-dataset -p {DATA_DIR} --unzip
    print("Done.")
else:
    print("FairFace already on Drive — skipping download.")

In [ ]:
import os
import itertools
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
import matplotlib.pyplot as plt
import matplotlib.cm as mpl_cm
import sklearn.metrics as skmetrics

print(f"TensorFlow {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
DATA_DIR   = '/content/drive/MyDrive/fairface'
TRAIN_IMG  = f'{DATA_DIR}/train'
VAL_IMG    = f'{DATA_DIR}/val'
TRAIN_CSV  = f'{DATA_DIR}/fairface_label_train.csv'
VAL_CSV    = f'{DATA_DIR}/fairface_label_val.csv'
MODEL_PATH = f'{DATA_DIR}/efficientnet_fairface.keras'

# FairFace's 7 racial groups — order defines the integer label index (0–6)
CLASSES = [
    'Black', 'East Asian', 'Indian',
    'Latino_Hispanic', 'Middle Eastern', 'Southeast Asian', 'White'
]
NUM_CLASSES  = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

IMG_SIZE   = 224   # EfficientNetB0 expected input resolution
BATCH_SIZE = 64
EPOCHS     = 30

## 1 · Data

FairFace provides two CSVs (`fairface_label_train.csv`, `fairface_label_val.csv`) that map each image filename to its race, age, and gender label. Training split: ~86,000 images. Validation: ~13,000.

### Why `tf.data` instead of `ImageDataGenerator`?
The original project used `ImageDataGenerator` — the old Keras API. `tf.data` is the modern replacement and is significantly faster:
- **Parallel loading** — reads the next batch from disk while the GPU is training on the current one
- **GPU-side augmentation** — random flips and brightness changes run in-graph rather than on the CPU
- **No memory ceiling** — streams from disk; the full dataset never has to fit in RAM

### Data augmentation
During training only, we apply:
- **RandomFlip (horizontal)** — a face mirrored left-to-right is still a valid face; this doubles effective variety
- **RandomBrightness / RandomContrast** — simulates different lighting conditions so the model generalizes

The validation set is **never augmented** — we want to measure performance on clean, unmodified images.

In [ ]:
def load_df(csv_path, img_dir):
    df = pd.read_csv(csv_path)
    # 'file' column contains paths like 'train/00001.jpg' — strip the subfolder prefix
    df['path']  = df['file'].apply(lambda f: os.path.join(img_dir, os.path.basename(f)))
    df['label'] = df['race'].map(CLASS_TO_IDX)
    df = df.dropna(subset=['label'])
    df['label'] = df['label'].astype(int)
    return df

train_df = load_df(TRAIN_CSV, TRAIN_IMG)
val_df   = load_df(VAL_CSV,   VAL_IMG)

print(f"Train: {len(train_df):,} images")
print(f"Val:   {len(val_df):,} images\n")
print("Class distribution (train):")
print(train_df['race'].value_counts().to_string())

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomBrightness(factor=0.1),
    layers.RandomContrast(factor=0.1),
], name='augmentation')

def parse_image(path, label):
    """Read a JPEG and resize to 224×224. EfficientNet handles normalization internally."""
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return img, label

def make_dataset(df, training=False):
    ds = tf.data.Dataset.from_tensor_slices((df['path'].values, df['label'].values))
    ds = ds.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(buffer_size=8000)
        ds = ds.map(lambda x, y: (augment(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds   = make_dataset(val_df,   training=False)
print("Datasets ready.")

In [ ]:
def show_samples(df, n_per_class=3):
    """Show a grid of sample images — rows = classes, columns = individual samples."""
    fig, axes = plt.subplots(NUM_CLASSES, n_per_class,
                             figsize=(n_per_class * 2.5, NUM_CLASSES * 2.5))
    for i, cls in enumerate(CLASSES):
        samples = df[df['race'] == cls].sample(n_per_class, random_state=42)
        for j, (_, row) in enumerate(samples.iterrows()):
            axes[i, j].imshow(plt.imread(row['path']))
            axes[i, j].axis('off')
        axes[i, 0].set_ylabel(cls, fontsize=9, rotation=0, labelpad=55, va='center')
    plt.suptitle('FairFace — Sample Images per Class', fontsize=12)
    plt.tight_layout()
    plt.show()

show_samples(train_df)

## 2 · Model — Transfer Learning with EfficientNetB0

### Why not train from scratch?
The original project used a 2-layer CNN trained from scratch. With only 7,000 images it never had enough signal to learn meaningful facial features — hence the poor results. Training a good CNN from scratch typically requires millions of images.

**Transfer learning** sidesteps this. EfficientNetB0 was pretrained on ImageNet (14M images, 1,000 categories). While learning to tell cats from cars, it developed edge detectors, texture filters, and shape recognizers that transfer directly to faces.

**Why EfficientNet over VGG16?** The original code used VGG16 preprocessing but never used VGG16 as an actual base. EfficientNet achieves better accuracy with far fewer parameters by scaling model depth, width, and resolution together using a compound coefficient. B0 is the smallest and fastest variant — still very capable for this task.

### Architecture

| Layer | Output shape | Role |
|---|---|---|
| Input | 224 × 224 × 3 | RGB image |
| EfficientNetB0 (frozen) | 7 × 7 × 1280 | Pretrained feature extractor |
| GlobalAveragePooling2D | 1280 | Collapses spatial dims to a single vector |
| Dropout(0.3) | 1280 | Randomly zeros 30% of values — prevents overfitting |
| Dense(7, softmax) | 7 | One probability per racial group |

### Two-phase training
- **Phase 1 — frozen base:** Only the Dense head trains. EfficientNet weights stay fixed. Fast convergence, no risk of destroying pretrained features.
- **Phase 2 — fine-tuning:** Unfreeze the top 20 layers of EfficientNet and train at a 100× smaller learning rate. Lets the pretrained features adapt slightly to faces specifically.

In [ ]:
def build_model():
    base = EfficientNetB0(
        include_top=False,          # remove ImageNet's 1000-class output head
        weights='imagenet',         # start from pretrained weights
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = False          # freeze for phase 1

    inputs  = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x       = base(inputs, training=False)  # training=False keeps BatchNorm in inference mode
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name='fairface_efficientnet')
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',  # expects integer labels, not one-hot
        metrics=['accuracy']
    )
    return model

model = build_model()
model.summary()

## 3 · Training

### Callbacks
Three callbacks control the training loop:

**EarlyStopping** — watches `val_loss`. If it doesn't improve for 5 consecutive epochs, training stops and the model rewinds to its best checkpoint automatically. Prevents wasting compute once the model starts overfitting.

**ReduceLROnPlateau** — if `val_loss` plateaus for 2 epochs, it halves the learning rate. Smaller learning rate = smaller weight updates, which often lets the optimizer find a better minimum instead of bouncing around it.

**ModelCheckpoint** — saves the best model to Drive after every epoch so you never lose progress if the Colab session disconnects.

### Loss function
The original project used `categorical_crossentropy`, which requires one-hot labels like `[0, 0, 1, 0, 0, 0, 0]`. We use `sparse_categorical_crossentropy` instead — same math, but accepts plain integers like `2`. Less memory, less preprocessing.

In [ ]:
callbacks_list = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=MODEL_PATH, monitor='val_accuracy', save_best_only=True, verbose=1
    ),
]

print("── Phase 1: training head only (EfficientNet frozen) ──")
history_1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks_list,
)

In [ ]:
# Unfreeze the top 20 layers of EfficientNetB0 for fine-tuning.
# Lower layers detect basic edges/textures — those don't need to change.
# Upper layers detect higher-level patterns — those benefit from adapting to faces.
base_model = model.layers[1]
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

# Recompile at a much smaller learning rate — large updates would
# destroy the pretrained weights we just unlocked.
model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"Trainable params after unfreezing: {sum(p.numpy().size for p in model.trainable_weights):,}")
print("── Phase 2: fine-tuning top layers ──")
history_2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks_list,
)

In [ ]:
def plot_training(h1, h2=None):
    acc   = h1.history['accuracy']     + (h2.history['accuracy']     if h2 else [])
    vacc  = h1.history['val_accuracy'] + (h2.history['val_accuracy'] if h2 else [])
    loss  = h1.history['loss']         + (h2.history['loss']         if h2 else [])
    vloss = h1.history['val_loss']     + (h2.history['val_loss']     if h2 else [])
    ep    = range(len(acc))
    split = len(h1.history['accuracy']) if h2 else None

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    for ax, train_v, val_v, title in [
        (ax1, acc,  vacc,  'Accuracy'),
        (ax2, loss, vloss, 'Loss'),
    ]:
        ax.plot(ep, train_v, label='Train')
        ax.plot(ep, val_v,   label='Validation')
        if split:
            ax.axvline(split, color='gray', linestyle='--', alpha=0.6, label='Fine-tune start')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.legend()

    plt.suptitle('Training History', fontsize=13)
    plt.tight_layout()
    plt.show()

plot_training(history_1, history_2)

## 4 · Bias Analysis

A single accuracy number hides everything that matters here. A model can be 75% accurate overall while being 95% accurate on one group and 40% on another — and the headline number would never tell you.

### Metrics

**Precision** — of all faces the model *predicted* as group X, what fraction actually were X? High precision = few false positives.

**Recall** — of all faces that *are* group X, what fraction did the model correctly identify? High recall = few false negatives (the model rarely misses real members of the group).

**F1 score** — harmonic mean of precision and recall. Penalizes imbalance between the two: a model that gets 100% precision but 0% recall scores F1 = 0.0, not 50%.

### Confusion matrix
Each **row** is the true class; each **column** is what the model predicted.
- Diagonal = correct predictions
- Off-diagonal = confusions between groups

The *pattern* of errors matters. If the model consistently misclassifies Group A as Group B but not the reverse, that asymmetry is a form of bias. Compare the confusion structure here to the original paper's Figure 4b — does a stronger model reduce bias evenly, or does it just shift where the errors land?

In [ ]:
print("Generating predictions on validation set...")
y_probs = model.predict(val_ds, verbose=1)
y_pred  = np.argmax(y_probs, axis=1)
y_true  = val_df['label'].values[:len(y_pred)]

report = skmetrics.classification_report(
    y_true, y_pred, target_names=CLASSES, output_dict=True
)
metrics_df = pd.DataFrame(report).T.loc[CLASSES, ['precision', 'recall', 'f1-score']]

fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(NUM_CLASSES)
w = 0.25
ax.bar(x - w, metrics_df['precision'], w, label='Precision', color='steelblue')
ax.bar(x,     metrics_df['recall'],    w, label='Recall',    color='coral')
ax.bar(x + w, metrics_df['f1-score'],  w, label='F1',        color='mediumseagreen')
ax.axhline(metrics_df['f1-score'].mean(), color='black', linestyle='--', alpha=0.5,
           label=f"Mean F1 ({metrics_df['f1-score'].mean():.2f})")
ax.set_xticks(x)
ax.set_xticklabels(CLASSES, rotation=30, ha='right')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Per-Class Performance — Bias Disparity View')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

print("\n", metrics_df.round(3).to_string())

In [ ]:
def plot_confusion_matrix(y_true, y_pred, classes):
    cm      = skmetrics.confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, data, title, fmt in [
        (axes[0], cm,      'Raw Counts',          'd'),
        (axes[1], cm_norm, 'Normalized (by row)', '.2f'),
    ]:
        im = ax.imshow(data, cmap='Blues')
        plt.colorbar(im, ax=ax)
        ax.set_xticks(range(len(classes)))
        ax.set_yticks(range(len(classes)))
        ax.set_xticklabels(classes, rotation=45, ha='right', fontsize=8)
        ax.set_yticklabels(classes, fontsize=8)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        ax.set_title(title)
        thresh = data.max() / 2
        for i, j in itertools.product(range(len(classes)), range(len(classes))):
            ax.text(j, i, format(data[i, j], fmt), ha='center', va='center',
                    fontsize=7, color='white' if data[i, j] > thresh else 'black')

    plt.suptitle('Confusion Matrix — EfficientNetB0 on FairFace', fontsize=12)
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(y_true, y_pred, CLASSES)

## 5 · Grad-CAM Visualizations

Accuracy numbers tell us *how much* the model errs. Grad-CAM (Gradient-weighted Class Activation Mapping) tells us *where it looks* when making a decision.

### How it works
1. Run a forward pass and record the activations of the last convolutional layer
2. Compute gradients of the predicted class score with respect to those activations — this measures "how much would this prediction change if each activation changed slightly?"
3. Average the gradients over spatial dimensions to get one importance weight per feature map
4. Take a weighted sum of the activation maps, apply ReLU (keep only positive contributions), and normalize
5. Resize to the original image size and overlay as a heatmap

**Warm colors (red/yellow)** = pixels that most increased the model's confidence.  
**Cool colors (blue)** = regions the model largely ignored.

### What to look for
- Does the model focus on facial structure (eyes, nose, jawline) or on skin tone and background?
- Are the attention patterns consistent across racial groups, or does the model use different cues for different classes?
- On misclassified images: what did it attend to that led it astray?

If the model focuses heavily on skin tone rather than structural features, that's evidence it learned a shortcut — exactly the kind of bias found in commercial systems like those reviewed by NIST in 2019.

In [ ]:
def get_gradcam(model, img_tensor):
    """
    Compute a Grad-CAM heatmap for the top predicted class.
    Returns (heatmap array, predicted class index).
    """
    base = model.layers[1]  # EfficientNetB0 submodel

    # Find the last Conv2D inside EfficientNetB0 — this is where spatial information
    # is richest before it gets collapsed by GlobalAveragePooling.
    last_conv = next(
        l for l in reversed(base.layers)
        if isinstance(l, tf.keras.layers.Conv2D)
    )

    # Split the model into two parts so we can watch gradients flow through conv outputs.
    conv_model = keras.Model(base.input, last_conv.output)

    head_input = keras.Input(shape=conv_model.output.shape[1:])
    x = head_input
    for layer in model.layers[2:]:  # GlobalAveragePooling → Dropout → Dense
        x = layer(x)
    head_model = keras.Model(head_input, x)

    with tf.GradientTape() as tape:
        conv_out = conv_model(img_tensor)
        tape.watch(conv_out)
        preds    = head_model(conv_out)
        pred_idx = int(tf.argmax(preds[0]))
        score    = preds[:, pred_idx]

    grads   = tape.gradient(score, conv_out)         # shape: (1, h, w, channels)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))  # one weight per channel
    cam     = tf.reduce_sum(conv_out[0] * weights, axis=-1)
    cam     = tf.nn.relu(cam)
    cam     = cam / (tf.reduce_max(cam) + 1e-8)
    return cam.numpy(), pred_idx


def overlay_heatmap(img_path, heatmap, alpha=0.45):
    img    = plt.imread(img_path).astype(float)
    h, w   = img.shape[:2]
    heat   = tf.image.resize(heatmap[..., tf.newaxis], [h, w]).numpy().squeeze()
    color  = mpl_cm.get_cmap('jet')(heat)[:, :, :3] * 255
    blend  = np.clip(img * (1 - alpha) + color * alpha, 0, 255).astype('uint8')
    return blend


def show_gradcam(df, model, n_per_class=2):
    cols = n_per_class * 2  # original + heatmap overlay per sample
    fig, axes = plt.subplots(NUM_CLASSES, cols,
                             figsize=(cols * 2.5, NUM_CLASSES * 2.8))
    fig.suptitle("Grad-CAM — What the model attends to per class\n"
                 "(left: original  |  right: heatmap overlay)", fontsize=11)

    for i, cls in enumerate(CLASSES):
        samples = df[df['race'] == cls].sample(n_per_class, random_state=7)
        col = 0
        for _, row in samples.iterrows():
            img_tensor = tf.expand_dims(
                tf.image.resize(
                    tf.image.decode_jpeg(tf.io.read_file(row['path']), channels=3),
                    [IMG_SIZE, IMG_SIZE]
                ), 0
            )
            heatmap, pred = get_gradcam(model, img_tensor)
            overlay = overlay_heatmap(row['path'], heatmap)
            correct = (pred == CLASS_TO_IDX[cls])

            axes[i, col].imshow(plt.imread(row['path']))
            axes[i, col].axis('off')
            axes[i, col + 1].imshow(overlay)
            axes[i, col + 1].set_title(f"→ {CLASSES[pred]}",
                                        fontsize=7,
                                        color='green' if correct else 'red')
            axes[i, col + 1].axis('off')
            col += 2

        axes[i, 0].set_ylabel(cls, fontsize=8, rotation=0, labelpad=60, va='center')

    plt.tight_layout()
    plt.show()

show_gradcam(val_df, model)